# Final Evaluation

- Goal: refit the shortlisted models, evaluate them once on the untouched test set, and decide which final model should be carried forward.


In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


## 1. Load Prepared Data

- Load the prepared dataset and apply the same deduplication choice used in the earlier notebooks.


In [2]:
data_path = "../dataset/creditcard.csv"
df = pd.read_csv(data_path)

original_shape = df.shape
df = df.drop_duplicates().reset_index(drop=True)
deduplicated_shape = df.shape

print("Original shape:", original_shape)
print("Deduplicated shape:", deduplicated_shape)
print("Duplicates removed:", original_shape[0] - deduplicated_shape[0])


Original shape: (284807, 31)
Deduplicated shape: (283726, 31)
Duplicates removed: 1081


## 2. Recreate the Split

- Recreate the same `75 / 15 / 10` split used earlier so the test set stays aligned with the previous workflow.
- After model selection is complete, combine `train + validation` and keep `test` untouched for final evaluation.


In [3]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=15 / 90,
    stratify=y_temp,
    random_state=42
)

X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)
print("Final training shape:", X_train_final.shape)

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))


Train shape: (212794, 30)
Validation shape: (42559, 30)
Test shape: (28373, 30)
Final training shape: (255353, 30)

Test class distribution:
Class
0    0.998343
1    0.001657
Name: proportion, dtype: float64


## 3. Finalists and Frozen Settings

- Freeze the shortlisted models, thresholds, and tuned hyperparameters from the earlier notebooks.
- In this notebook, we do not tune anything further. We only refit and evaluate on the test set.


In [4]:
def build_logistic_baseline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(random_state=42, max_iter=1000))
    ])


def build_tuned_random_forest():
    return RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        n_estimators=200,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        max_depth=20,
        class_weight="balanced"
    )


def build_tuned_xgboost():
    return XGBClassifier(
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
        subsample=0.8,
        scale_pos_weight=10,
        n_estimators=200,
        min_child_weight=5,
        max_depth=8,
        learning_rate=0.1,
        gamma=0,
        colsample_bytree=0.7
    )


def build_tuned_lightgbm():
    return LGBMClassifier(
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        subsample=1.0,
        scale_pos_weight=1,
        reg_lambda=1.0,
        reg_alpha=0.1,
        num_leaves=63,
        n_estimators=500,
        min_child_samples=100,
        max_depth=-1,
        learning_rate=0.1,
        colsample_bytree=0.7
    )


FINAL_MODEL_CONFIGS = {
    "Logistic Regression Baseline": {
        "builder": build_logistic_baseline,
        "threshold": 0.10,
        "selection_reason": "Baseline reference from the training notebook",
        "hyperparameters": {
            "random_state": 42,
            "max_iter": 1000,
            "scaler": "StandardScaler"
        }
    },
    "Tuned Random Forest": {
        "builder": build_tuned_random_forest,
        "threshold": 0.30,
        "selection_reason": "Best Random Forest family candidate",
        "hyperparameters": {
            "n_estimators": 200,
            "min_samples_split": 5,
            "min_samples_leaf": 2,
            "max_features": "sqrt",
            "max_depth": 20,
            "class_weight": "balanced",
            "random_state": 42,
            "n_jobs": -1
        }
    },
    "Tuned XGBoost": {
        "builder": build_tuned_xgboost,
        "threshold": 0.50,
        "selection_reason": "Best PR-AUC overall on validation",
        "hyperparameters": {
            "subsample": 0.8,
            "scale_pos_weight": 10,
            "n_estimators": 200,
            "min_child_weight": 5,
            "max_depth": 8,
            "learning_rate": 0.1,
            "gamma": 0,
            "colsample_bytree": 0.7,
            "random_state": 42,
            "eval_metric": "logloss",
            "n_jobs": -1
        }
    },
    "Tuned LightGBM": {
        "builder": build_tuned_lightgbm,
        "threshold": 0.50,
        "selection_reason": "Best F1 overall and second-best PR-AUC on validation",
        "hyperparameters": {
            "subsample": 1.0,
            "scale_pos_weight": 1,
            "reg_lambda": 1.0,
            "reg_alpha": 0.1,
            "num_leaves": 63,
            "n_estimators": 500,
            "min_child_samples": 100,
            "max_depth": -1,
            "learning_rate": 0.1,
            "colsample_bytree": 0.7,
            "random_state": 42,
            "n_jobs": -1,
            "verbose": -1
        }
    }
}

finalists_config_df = pd.DataFrame([
    {
        "Model": model_name,
        "Threshold": config["threshold"],
        "Selection Reason": config["selection_reason"],
        "Hyperparameters": str(config["hyperparameters"])
    }
    for model_name, config in FINAL_MODEL_CONFIGS.items()
])

finalists_config_df


,Model,Threshold,Selection Reason,Hyperparameters
0,Logistic Regression Baseline,0.1,Baseline reference from the training notebook,"{'random_state': 42, 'max_iter': 1000, 'scaler..."
1,Tuned Random Forest,0.3,Best Random Forest family candidate,"{'n_estimators': 200, 'min_samples_split': 5, ..."
2,Tuned XGBoost,0.5,Best PR-AUC overall on validation,"{'subsample': 0.8, 'scale_pos_weight': 10, 'n_..."
3,Tuned LightGBM,0.5,Best F1 overall and second-best PR-AUC on vali...,"{'subsample': 1.0, 'scale_pos_weight': 1, 'reg..."


## 4. Refit and Evaluate in a Loop

- Fit each frozen model on `train + validation`.
- Use the frozen validation threshold for the final test-set predictions.
- This is where the dictionary-based loop helps, because the evaluation logic is the same across models.


In [5]:
trained_models = {}
test_results = []

for model_name, config in FINAL_MODEL_CONFIGS.items():
    model = config["builder"]()
    model.fit(X_train_final, y_train_final)

    trained_models[model_name] = model

    y_score_test = model.predict_proba(X_test)[:, 1]
    threshold = config["threshold"]
    y_pred_test = (y_score_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

    test_results.append({
        "Model": model_name,
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision_score(y_test, y_pred_test, zero_division=0),
        "Recall": recall_score(y_test, y_pred_test, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred_test, zero_division=0),
        "PR-AUC": average_precision_score(y_test, y_score_test),
        "ROC-AUC": roc_auc_score(y_test, y_score_test)
    })

final_test_results_df = pd.DataFrame(test_results).sort_values(
    by=["PR-AUC", "F1 Score"],
    ascending=False
).reset_index(drop=True)

final_test_results_df.round(4)


,Model,Threshold,TN,FP,FN,TP,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
0,Tuned XGBoost,0.5,28324,2,7,40,0.9524,0.8511,0.8989,0.8967,0.9866
1,Tuned LightGBM,0.5,28324,2,8,39,0.9512,0.8298,0.8864,0.8922,0.9840
2,Tuned Random Forest,0.3,28322,4,6,41,0.9111,0.8723,0.8913,0.8860,0.9655
3,Logistic Regression Baseline,0.1,28316,10,7,40,0.8000,0.8511,0.8247,0.7452,0.9674


## 5. Final Comparison Tables

- Review the final comparison from two angles: threshold-based performance and threshold-free ranking performance.


In [6]:
final_test_results_df[
    ["Model", "Threshold", "Precision", "Recall", "F1 Score", "PR-AUC", "ROC-AUC"]
].round(4)


,Model,Threshold,Precision,Recall,F1 Score,PR-AUC,ROC-AUC
0,Tuned XGBoost,0.5,0.9524,0.8511,0.8989,0.8967,0.9866
1,Tuned LightGBM,0.5,0.9512,0.8298,0.8864,0.8922,0.9840
2,Tuned Random Forest,0.3,0.9111,0.8723,0.8913,0.8860,0.9655
3,Logistic Regression Baseline,0.1,0.8000,0.8511,0.8247,0.7452,0.9674


In [7]:
final_test_results_df[
    ["Model", "PR-AUC", "ROC-AUC"]
].sort_values(
    by="PR-AUC",
    ascending=False
).round(4)


,Model,PR-AUC,ROC-AUC
0,Tuned XGBoost,0.8967,0.9866
1,Tuned LightGBM,0.8922,0.9840
2,Tuned Random Forest,0.8860,0.9655
3,Logistic Regression Baseline,0.7452,0.9674


## 6. Error Analysis

- Focus the error analysis on the final winning model from the test set.
- Look at false positives and false negatives separately so the remaining mistakes are easier to understand.


In [10]:
best_model_name = final_test_results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_threshold = FINAL_MODEL_CONFIGS[best_model_name]["threshold"]

best_model_score_test = best_model.predict_proba(X_test)[:, 1]
best_model_pred_test = (best_model_score_test >= best_threshold).astype(int)

best_model_test_analysis_df = X_test.reset_index(drop=True).copy()
best_model_test_analysis_df["y_true"] = y_test.reset_index(drop=True)
best_model_test_analysis_df["predicted_score"] = best_model_score_test
best_model_test_analysis_df["predicted_class"] = best_model_pred_test

conditions = [
    (best_model_test_analysis_df["y_true"] == 0) & (best_model_test_analysis_df["predicted_class"] == 0),
    (best_model_test_analysis_df["y_true"] == 0) & (best_model_test_analysis_df["predicted_class"] == 1),
    (best_model_test_analysis_df["y_true"] == 1) & (best_model_test_analysis_df["predicted_class"] == 0),
    (best_model_test_analysis_df["y_true"] == 1) & (best_model_test_analysis_df["predicted_class"] == 1)
]

choices = ["True Negative", "False Positive", "False Negative", "True Positive"]

best_model_test_analysis_df["Outcome"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

error_summary_df = best_model_test_analysis_df["Outcome"].value_counts().rename_axis("Outcome").reset_index(name="Count")
error_summary_df["Proportion"] = error_summary_df["Count"] / len(best_model_test_analysis_df)

false_positives_df = best_model_test_analysis_df[best_model_test_analysis_df["Outcome"] == "False Positive"].copy()
false_negatives_df = best_model_test_analysis_df[best_model_test_analysis_df["Outcome"] == "False Negative"].copy()

error_summary_df.round(4)


,Outcome,Count,Proportion
0,True Negative,28324,0.9983
1,True Positive,40,0.0014
2,False Negative,7,0.0002
3,False Positive,2,0.0001


In [11]:
error_subset_summary_df = pd.DataFrame([
    {
        "Subset": "False Positives",
        "Count": len(false_positives_df),
        "Average Score": false_positives_df["predicted_score"].mean(),
        "Average Amount": false_positives_df["Amount"].mean(),
        "Median Amount": false_positives_df["Amount"].median(),
        "Average Time": false_positives_df["Time"].mean()
    },
    {
        "Subset": "False Negatives",
        "Count": len(false_negatives_df),
        "Average Score": false_negatives_df["predicted_score"].mean(),
        "Average Amount": false_negatives_df["Amount"].mean(),
        "Median Amount": false_negatives_df["Amount"].median(),
        "Average Time": false_negatives_df["Time"].mean()
    }
])

error_subset_summary_df.round(4)


,Subset,Count,Average Score,Average Amount,Median Amount,Average Time
0,False Positives,2,0.8652,12900.5300,12900.53,120952.0
1,False Negatives,7,0.0652,90.3971,1.18,69244.0


## 7. Model Interpretability

- Use feature importance from the final winning model to see which inputs are driving predictions most strongly.
- Treat this as directional model interpretation, not as proof of causality.


In [12]:
if hasattr(best_model, "feature_importances_"):
    importance_values = best_model.feature_importances_
    importance_method = "feature_importances_"
elif isinstance(best_model, Pipeline) and "model" in best_model.named_steps and hasattr(best_model.named_steps["model"], "coef_"):
    importance_values = np.abs(best_model.named_steps["model"].coef_[0])
    importance_method = "absolute_coefficient"
else:
    raise ValueError("This model does not expose a simple built-in importance method.")

best_model_feature_importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": importance_values
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

top_feature_importance_df = best_model_feature_importance_df.head(15).copy()
top_feature_importance_df["Importance Method"] = importance_method

top_feature_importance_df.round(6)


,Feature,Importance,Importance Method
0,V14,0.389940,feature_importances_
1,V17,0.166541,feature_importances_
2,V10,0.086823,feature_importances_
3,V12,0.035082,feature_importances_
4,V4,0.024384,feature_importances_
5,V7,0.021332,feature_importances_
6,V11,0.018120,feature_importances_
7,V18,0.017191,feature_importances_
8,V16,0.014369,feature_importances_
9,V8,0.013724,feature_importances_


## 8. Final Test Observations

- `Tuned XGBoost` is the final winner on the test set and remains the strongest model by the primary metric, with `PR-AUC = 0.8967`.
- `Tuned XGBoost` also gives the best overall threshold-based result among the finalists here, with `Precision = 0.9524`, `Recall = 0.8511`, and `F1 Score = 0.8989` at threshold `0.5`.
- `Tuned LightGBM` is a very close second and confirms that the boosting family is the strongest direction for this project.
- `Tuned Random Forest` remains highly competitive and achieves the highest recall among the finalists, but it falls slightly behind `Tuned XGBoost` on both `PR-AUC` and `F1 Score`.
- `Logistic Regression Baseline` is clearly weaker than the advanced finalists on `PR-AUC`, which shows that the later model exploration produced a real improvement rather than a cosmetic one.
- The final XGBoost test errors are small in absolute terms: `2` false positives and `7` false negatives.
- The false positives have very high predicted scores on average and very large transaction amounts, which suggests the model is treating a few extreme legitimate transactions as strongly fraud-like.
- The false negatives have much lower predicted scores on average and much smaller amounts, which suggests some missed frauds are more subtle and less separable from normal behavior.
- The strongest feature-importance signals in the final model come from `V14`, `V17`, and `V10`, with `V14` standing out clearly above the rest.
- `Amount` does contribute to the final model, but it is not one of the dominant drivers compared with the strongest transformed PCA-style features.
- The final decision is therefore straightforward: `Tuned XGBoost` should be carried forward as the main model, while `Tuned LightGBM` is the strongest backup candidate.


## 9. Export Useful Evaluation Outputs

- Save the frozen model settings and final test results for later review.


In [8]:
confusion_matrix_rows = []
test_predictions_export_df = pd.DataFrame({
    "y_true": y_test.reset_index(drop=True)
})

for model_name, model in trained_models.items():
    threshold = FINAL_MODEL_CONFIGS[model_name]["threshold"]
    y_score_test = model.predict_proba(X_test)[:, 1]
    y_pred_test = (y_score_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

    confusion_matrix_rows.append({
        "Model": model_name,
        "Threshold": threshold,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

    safe_name = model_name.lower().replace(" ", "_")
    test_predictions_export_df[f"{safe_name}_score"] = y_score_test
    test_predictions_export_df[f"{safe_name}_pred"] = y_pred_test

final_confusion_matrix_df = pd.DataFrame(confusion_matrix_rows).sort_values(
    by="Model"
).reset_index(drop=True)

final_confusion_matrix_df


,Model,Threshold,TN,FP,FN,TP
0,Logistic Regression Baseline,0.1,28316,10,7,40
1,Tuned LightGBM,0.5,28324,2,8,39
2,Tuned Random Forest,0.3,28322,4,6,41
3,Tuned XGBoost,0.5,28324,2,7,40


In [9]:
outputs_dir = "../outputs/evaluate"
os.makedirs(outputs_dir, exist_ok=True)

finalists_config_df.to_csv(f"{outputs_dir}/finalists_for_evaluation.csv", index=False)
final_test_results_df.to_csv(f"{outputs_dir}/final_test_results.csv", index=False)
error_summary_df.to_csv(f"{outputs_dir}/best_model_error_summary.csv", index=False)
error_subset_summary_df.to_csv(f"{outputs_dir}/best_model_error_subset_summary.csv", index=False)
false_positives_df.to_csv(f"{outputs_dir}/best_model_false_positives.csv", index=False)
false_negatives_df.to_csv(f"{outputs_dir}/best_model_false_negatives.csv", index=False)
best_model_feature_importance_df.to_csv(f"{outputs_dir}/best_model_feature_importance.csv", index=False)
top_feature_importance_df.to_csv(f"{outputs_dir}/best_model_top_feature_importance.csv", index=False)
final_confusion_matrix_df.to_csv(f"{outputs_dir}/final_confusion_matrices.csv", index=False)
test_predictions_export_df.to_csv(f"{outputs_dir}/test_set_model_predictions.csv", index=False)

best_model_row = final_test_results_df.iloc[0]
best_model_summary_df = final_test_results_df.head(1).copy()
best_model_summary_df.to_csv(f"{outputs_dir}/best_model_summary.csv", index=False)

with open(f"{outputs_dir}/evaluation_notes.txt", "w") as f:
    f.write("Final Evaluation Summary\n")
    f.write("=======================\n")
    f.write(f"Best model by PR-AUC on test: {best_model_row['Model']}\n")
    f.write(f"Threshold used: {best_model_row['Threshold']}\n")
    f.write(f"PR-AUC: {best_model_row['PR-AUC']:.6f}\n")
    f.write(f"ROC-AUC: {best_model_row['ROC-AUC']:.6f}\n")
    f.write(f"Precision: {best_model_row['Precision']:.6f}\n")
    f.write(f"Recall: {best_model_row['Recall']:.6f}\n")
    f.write(f"F1 Score: {best_model_row['F1 Score']:.6f}\n")
    f.write("\nAll finalists ranked by PR-AUC:\n")
    for _, row in final_test_results_df.iterrows():
        f.write(
            f"- {row['Model']}: threshold={row['Threshold']}, PR-AUC={row['PR-AUC']:.6f}, ROC-AUC={row['ROC-AUC']:.6f}, Precision={row['Precision']:.6f}, Recall={row['Recall']:.6f}, F1={row['F1 Score']:.6f}\n"
        )

print("Saved evaluation outputs to:", outputs_dir)


Saved evaluation outputs to: ../outputs/evaluate
